Troubleshooting:

get_beams calls self.compute_single_model which is returning beams=False
get_beams the indexing into beams and errors out

In [1]:
# Imports and cd fits directory
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits, ascii
import drizzlepac
import glob

import grizli
from grizli import utils
from grizli import multifit
from grizli.multifit import GroupFLT

import os
os.chdir("/Users/keith/astr/research_astr/summer-roman-project/FOV0_extraction/fits")



The following task in the stsci.skypac package can be run with TEAL:
                                    skymatch                                    
The following tasks in the drizzlepac package can be run with TEAL:
    astrodrizzle       config_testbed      imagefindpars           mapreg       
       photeq            pixreplace           pixtopix            pixtosky      
  refimagefindpars       resetbits          runastrodriz          skytopix      
     tweakback            tweakreg           updatenpol


In [2]:
grism_files = glob.glob("model_slitless*.fits")
catalog = "seg_cat.detect.cat"

In [3]:
grp = GroupFLT(grism_files=grism_files, 
                        catalog=catalog, 
                        cpu_count=-1, sci_extn=1, pad=256)

Files loaded - 0.44 sec.


In [4]:
### Find IDs of specific objects to extract
import astropy.units as u
tab = utils.GTable()
tab['ra'] = [53.24318031117099, 53.24775107767136]
tab['dec'] = [-27.82854257538046, -53.24775107767136]

idx, dr = grp.catalog.match_to_catalog_sky(tab)
source_ids = grp.catalog['NUMBER'][idx]
tab['id'] = source_ids
tab['dr'] = dr.to(u.mas)
tab['dr'].format='.1f'

In [5]:
tab['id']

7409
6926


In [6]:
grp.get_beams(7409, size=77, show_exception=True)


########################################## 
# ! Exception (2024-06-27 21:12:33.362)
#
# !Traceback (most recent call last):
# !  File "/Users/keith/miniconda3/envs/grizli_env/lib/python3.12/site-packages/grizli/multifit.py", line 709, in get_beams
# !    out_beam = model.BeamCutout(flt=flt, beam=beam[beam_id],
# !                                              ~~~~^^^^^^^^^
# !TypeError: 'bool' object is not subscriptable
# !
######################################### 



########################################## 
# ! Exception (2024-06-27 21:12:33.363)
#
# !Traceback (most recent call last):
# !  File "/Users/keith/miniconda3/envs/grizli_env/lib/python3.12/site-packages/grizli/multifit.py", line 709, in get_beams
# !    out_beam = model.BeamCutout(flt=flt, beam=beam[beam_id],
# !                                              ~~~~^^^^^^^^^
# !TypeError: 'bool' object is not subscriptable
# !
######################################### 




[]

In [7]:
id = source_ids[0]

# Pull out the 2D cutouts
beams = grp.get_beams(id, size=80)
mb = multifit.MultiBeam(beams, fcontam=0.2, psf=False)

# Save a FITS file with the 2D cutouts (beams) from the individual exposures
mb.write_master_fits()

# Fit polynomial model for initial continuum subtraction
wave = np.linspace(2000,2.5e4,100)
poly_templates = grizli.utils.polynomial_templates(wave, order=7)
pfit = mb.template_at_z(z=0, templates=poly_templates, fit_background=True, 
                        fitter='lstsq', get_uncertainties=2)

# Drizzle grisms / PAs and make a figure
hdu, fig = mb.drizzle_grisms_and_PAs(fcontam=0.2, flambda=False, kernel='point', 
                                     size=32, zfit=pfit, diff=True)

IndexError: list index out of range